# Proving Audio Frequency Visualization Works

Our extension uses the Web Audio API's `AnalyserNode` to generate a frequency visualization. This notebook proves the underlying mathematics (Fast Fourier Transform or FFT) that powers that visualizer.

**Improvement Applied:** The raw FFT math calculates magnitude. But browsers map this mathematically to a decibel range (e.g. -100dB to -30dB) and then convert it into an 8-bit unsigned integer (0-255 scale) before JavaScript receives it. We have updated this notebook to perform the exact same mapping so it behaves exactly like the web extension.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Setup sampling rate and time axis
sample_rate = 44100  # Standard audio sampling rate
duration = 1.0       # 1 second of audio
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

In [ ]:
# Generate a low frequency (440 Hz) and a high frequency (4000 Hz) wave
freq1 = 440.0
freq2 = 4000.0

wave1 = 0.5 * np.sin(2 * np.pi * freq1 * t)
wave2 = 0.3 * np.sin(2 * np.pi * freq2 * t)

# Combine them
audio_signal = wave1 + wave2

### Perform Fast Fourier Transform (FFT) and Map to Web Audio API Scale
This perfectly simulates `AnalyserNode.getByteFrequencyData()`.

In [ ]:
# Compute FFT
fft_result = np.fft.fft(audio_signal)
fft_magnitude = np.abs(fft_result)

frequencies = np.fft.fftfreq(len(fft_result), 1/sample_rate)

half_n = len(frequencies) // 2
frequencies = frequencies[:half_n]
fft_magnitude = fft_magnitude[:half_n]

# === NEW IMPROVEMENT: Web Audio API Uint8 Conversion ===
# Convert magnitude to Decibels (dB)
fft_db = 20 * np.log10(fft_magnitude + 1e-10)

# Normalize to 0-255 scale (matching Web Audio API's AnalyserNode Uint8Array output)
min_db = -100
max_db = -30

# Clamp and scale values to exactly match how Chrome renders the audio array
fft_normalized = np.clip((fft_db - min_db) / (max_db - min_db), 0, 1) * 255
fft_uint8 = fft_normalized.astype(np.uint8)

# Plot the Web-Audio equivalent scale
plt.figure(figsize=(10, 4))
plt.plot(frequencies, fft_uint8, color='purple')
plt.title("Frequency Spectrum (Exact Web Audio API 0-255 Scale)")
plt.xlabel("Frequency [Hz]")
plt.ylabel("Uint8 Value (0-255)")
plt.xlim(0, 5000)
plt.grid(True)
plt.show()